# 0055 step 1: the Wikidata dumps, downloaded once and counted

**Part A** is Wikimedia's JSON dump (done 2026-11-06; its download check is run again after a Drive lag stopped it).
**Part B** is 0055's source dump since Cowork's addendum: the WikiKGQA 2026 challenge's own Wikidata (Turtle, 121 GB,
CC0, Zenodo record 22661413; Nick's yes). B1 is done (113 parts, the md5 matches Zenodo's). B2 stalled four times at
the dump's very end: its forked workers held lbzip2's input pipe open, so lbzip2 never saw the end of its input. The
workers now start from a forkserver; B2 first runs the check that the pipe ends (a few seconds), then resumes from its
checkpoint (920.6 GB, 2026-11-09). **Restart the session (Runtime > Restart session) so no old count runs underneath,
run the first cell (it pulls the fix), then B2.**

**Runtime:** CPU, high RAM (no GPU). **Sign in to the account whose Drive has the room** (the 5 TB one): the dump
(about 104 GB, bz2) is kept there as the source copy, in 1 GB parts.

1. **Download:** two connections at a time (Wikimedia's limit), in parts. If the runtime drops, run the cell again: it
   resumes at the first missing part. At the end the parts' sha1 is checked against Wikimedia's (a part the Drive
   shows late after its rename is waited for; if the check still stops on a missing part, run the cell again).
2. **Count:** one pass over the parts, never unpacked to disk: the counts (`counts.json`) and the 1% sample
   (`sample_1pct.jsonl.gz`), written to the Drive. **If the runtime drops here, run the cell again:** it resumes from
   its last checkpoint (`resume.json`, every 15 minutes; the sample is kept in closed segments meanwhile). A resumed
   pass first decompresses up to the checkpoint without counting (faster than counting), then goes on. The pass
   hashes the parts too (checked against the download's `manifest.json` when there is one) and stops if a part is
   missing or corrupt.

Keep this tab open while it runs. **Disconnect when the count prints its summary.** Then tell Claude Code.

In [ ]:
import os, shutil, subprocess, sys
if not os.path.exists('/content/CubbyLLM'):
    !git clone -q https://github.com/Grillcheese-AI/CubbyLLM.git /content/CubbyLLM
else:
    !cd /content/CubbyLLM && git pull -q --ff-only
!apt-get -qq install -y lbzip2 > /dev/null
!pip -q install orjson
from google.colab import drive; drive.mount('/content/drive')
REPO = '/content/CubbyLLM'
HOME = '/content/drive/MyDrive/wikidata'                 # a folder at the Drive's root (Nick)
PARTS, COUNT = f'{HOME}/dump_parts', f'{HOME}/count'
os.makedirs(PARTS, exist_ok=True); os.makedirs(COUNT, exist_ok=True)
free = shutil.disk_usage('/content/drive').free / 1e9
print(f'Drive free: {free:.0f} GB; cores: {os.cpu_count()}; lbzip2: {bool(shutil.which("lbzip2"))}')
assert free > 250, 'this Drive is too small: sign in to the account with the room'

def step(script, *args):
    cmd = [sys.executable, '-u', f'validation/fact_store/{script}', *args]
    with open(f'{HOME}/{script}.log', 'a', encoding='utf-8', buffering=1) as f:
        p = subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        try:
            for line in p.stdout:
                print(line, end=''); f.write(line)
        except KeyboardInterrupt:                        # a stopped cell stops its script too (it ran on, unseen)
            p.kill(); p.wait(); print(f'{script} stopped'); raise
        assert p.wait() == 0, f'{script} failed ({p.returncode})'

## A1. Wikimedia's JSON: download (resumable: run again if the runtime drops)

In [ ]:
step('dump_fetch.py', '--out', PARTS, '--stage', '/content/stage')
print(open(f'{PARTS}/manifest.json').read())

## A2. Wikimedia's JSON: count, and the 1% sample (done: skip)

In [ ]:
step('dump_count.py', '--parts', PARTS, '--out', COUNT, '--workers', str(os.cpu_count()), '--stage', '/content/stage_count')

## B1. The WikiKGQA dump (Turtle): download, checked against Zenodo's md5 (resumable)

In [ ]:
KPARTS, KCOUNT = f'{HOME}/kgqa_parts', f'{HOME}/kgqa_count'
os.makedirs(KPARTS, exist_ok=True); os.makedirs(KCOUNT, exist_ok=True)
# each part is written to this runtime's own disk, checked, then copied whole to the Drive (the Drive's mount lost
# freshly written parts); 4 connections (Zenodo states no limit; Wikimedia's dump, A1, keeps 2)
step('dump_fetch.py', '--out', KPARTS, '--zenodo', '22661413:wikikgqa-wikidata-all-dump.ttl.bz2',
     '--stage', '/content/stage', '--workers', '4')
print(open(f'{KPARTS}/manifest.json').read())

## B2. The WikiKGQA dump: count, the 1% sample (the same ids as A2's), its date and its redirects (resumable: run again if the runtime drops)

In [ ]:
# the workers must not hold lbzip2's input pipe open (the stall at the dump's end): checked here, on Linux, first
chk = subprocess.run([sys.executable, '-m', 'pytest', '-p', 'no:cacheprovider',
                      'tests/validation/test_fact_store_pool.py'], cwd=REPO, capture_output=True, text=True)
print(chk.stdout[-600:]); assert chk.returncode == 0 and 'skipped' not in chk.stdout, 'the pool check failed: tell Claude Code'
step('ttl_count.py', '--parts', KPARTS, '--out', KCOUNT, '--workers', str(os.cpu_count()), '--stage', '/content/stage_count')

## Done

**Disconnect the runtime.** At this Drive's root: `wikidata/count/` and `wikidata/dump_parts/` (Part A),
`wikidata/kgqa_count/` and `wikidata/kgqa_parts/` (Part B). Then tell Claude Code.